# 04b — Social Media Charts (Pillow)
Publication-ready PNG charts using the @unwelcomedata brand palette.
All charts export to twitter_landscape (1600x900px) with watermark.

**Rendering engine:** Pillow (PIL) via shared/chart_factory.py — no browser dependency.

**Standard layout:**
- Title: bold, left-aligned (statement or question, no year/unit)
- Subtitle: unit or scope ("Rate per 100,000 population")
- Footer: source with year ("CDC WONDER 2024")
- Watermark: @unwelcomedata bottom-right

**Chart types available:**
-  — two horizontal bar panels on shared scale
-  — stacked horizontal bars with segments + optional inner segments
-  — single panel of ranked horizontal bars

**Data source:** All chart data stored in DuckDB  tables.


In [ ]:
# ===================================================================
# SETUP
# ===================================================================
import sys
import os
from pathlib import Path
import duckdb
import yaml

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / 'src'))
sys.path.insert(0, str(PROJECT.parent / 'shared'))

from chart_factory import render_chart

with open(PROJECT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Connect to DuckDB
conn = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'))

# Create outputs/social directory
social_dir = PROJECT / 'outputs' / 'social'
social_dir.mkdir(parents=True, exist_ok=True)

print('✓ chart_factory loaded (Pillow backend)')
print(f'✓ DuckDB connected')
print(f'✓ Charts export to: {social_dir}')


## Color Palette
Define project colors here. Re-run this cell to update all charts below.


In [ ]:
# ===================================================================
# COLOR PALETTE — edit per project
# ===================================================================
# See shared/viz.py for brand palette constants (BRAND, COOLORS, etc.)
# Define chart-specific colors here:

C_PRIMARY = '#0A9396'       # Teal
C_SECONDARY = '#EE9B00'     # Golden Orange
C_ACCENT = '#AE2012'        # Dark Red (highlight)

print('✓ Palette loaded')


---
## Chart 1: [Title]
Description of what this chart shows.


In [ ]:
# Example: side-by-side bars
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',
    # Data — DuckDB table names (must start with chart_)
    'table_left': 'chart_left_panel',
    'table_right': 'chart_right_panel',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',  # auto-generated if missing
    # Text (consistent pattern: title=statement, subtitle=unit, source=data+year)
    'title': 'Chart Title: Group A vs Group B',
    'subtitle': 'Rate per 100,000 population',
    'source': 'Data Source 2024',
    'left_title': 'Group A',
    'right_title': 'Group B',
    # Colors
    'left_color': C_PRIMARY,
    'right_color': C_SECONDARY,
    # Highlight (optional) — accent a specific bar
    # 'highlight_left': {'category': 'Name', 'color': C_ACCENT, 'annotation': '#N for other'},
    # 'highlight_right': {'category': 'Name', 'color': C_ACCENT},
    # Detail bars (optional) — rendered below the main chart
    # 'detail_bar': {'table': 'chart_detail_table', 'title': 'Breakdown:', 'gradient': ['#c1','#c2','#c3','#c4']},
    # Export
    'filename': '01_my_chart',
})


---
## Chart 2: [Title]
Description.


In [ ]:
# Example: stacked bars
render_chart({
    'type': 'stacked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',
    # Data
    'table': 'chart_stacked_data',
    'category_col': 'cause',
    # Segments — each is a column in the table
    'segments': [
        {'value_col': 'group_a', 'label': 'Group A', 'color': C_PRIMARY, 'text_color': '#003049'},
        {'value_col': 'group_b', 'label': 'Group B', 'color': C_SECONDARY, 'text_color': '#003049'},
    ],
    # Inner segments (optional) — subdivide a specific bar
    # 'inner_segments': {
    #     'category': 'BarName',  # which bar gets inner segments
    #     'table': 'chart_inner_detail',
    #     'text_color': '#E9D8A6',
    # },
    # Text
    'title': 'Chart Title',
    'subtitle': 'Scope or unit description',
    'source': 'Data Source 2024',
    # Export
    'filename': '02_my_stacked_chart',
})


---
## Summary & Cleanup


In [ ]:
conn.close()

# Verify all exports
pngs = sorted(social_dir.glob('*.png'))
print('=== ALL CHARTS COMPLETE ===')
print(f'\n✓ Generated {len(pngs)} publication-ready charts:')
for png in pngs:
    size_kb = png.stat().st_size / 1024
    print(f'  • {png.name} ({size_kb:.0f} KB)')
print('\nAll charts are twitter_landscape (1600x900px) with @unwelcomedata watermark.')
